# Module 4 - Knowledge Assistant corpus prep + KA build (fixed)
The KA itself is created via Agent Bricks UI. This notebook stages the corpus
and emits the click-through instructions.

In [ ]:
%run ./_config


## 1. Stage the corpus

In [ ]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {BASE}.ka_corpus
""")
spark.sql(f"""
LIST '{VOL_ADVISORIES}'
""")


In [ ]:
import shutil
import os

src_advisories = VOL_ADVISORIES
src_stigs = VOL_STIGS
dst = VOL_KA_CORPUS

for src in (src_advisories, src_stigs):
    if not os.path.exists(src):
        print(f"WARN: {src} does not exist, skipping")
        continue
    for fn in os.listdir(src):
        target = f"{dst}/{fn}"
        try:
            shutil.copy(f"{src}/{fn}", target)
            print(f"Copied {fn}")
        except Exception as e:
            print(f"WARN: failed to copy {fn}: {e}")

techniques = spark.table(f"{BASE}.attack_techniques").collect()
written = 0
for row in techniques[:50]:
    if not row.technique_id:
        continue
    fname = f"{dst}/attack_{row.technique_id}.txt"
    try:
        with open(fname, "w") as f:
            f.write(f"# {row.technique_id}: {row.name}\n\n{row.description}\n\nPlatforms: {', '.join(row.platforms or [])}")
        written += 1
    except Exception as e:
        print(f"WARN: failed to write {fname}: {e}")
print(f"Wrote {written} ATT&CK technique files to {dst}")


In [ ]:
print("KA corpus contents:")
for fn in os.listdir(dst):
    size = os.path.getsize(f"{dst}/{fn}")
    print(f"  {fn} ({size} bytes)")

## 2. Provision the Knowledge Assistant via API

This replaces the click-through (Agents > Knowledge Assistant > Create) with the Databricks SDK. The KA endpoint takes 10-20 minutes to warm up; we kick it off and let module 5 handle the case where it isn't yet ONLINE.


In [ ]:
KA_SYSTEM_PROMPT = """
You are a DISA cyber threat intelligence assistant. Answer questions about CISA advisories,
DoD STIGs, and MITRE ATT&CK techniques. Always cite the source document name and section.

If the answer is not in the retrieved context, say so explicitly. Do not make up CVE IDs,
STIG identifiers, or ATT&CK technique IDs.

When citing a STIG control, include both the STIG-ID (e.g., V-220701) and the severity (CAT I/II/III).
When citing an ATT&CK technique, include the technique ID (e.g., T1059.001).
""".strip()

# KA_NAME comes from _config
KA_VOLUME = VOL_KA_CORPUS
KA_BASE = "/api/2.1/knowledge-assistants"

from databricks.sdk import WorkspaceClient
import time
w = WorkspaceClient()

# Look for an existing KA by display_name (idempotent re-runs).
KA_ID = None
try:
    res = w.api_client.do('GET', KA_BASE + '?page_size=100')
    for ka in (res.get('knowledge_assistants') or []):
        if ka.get('display_name') == KA_NAME:
            KA_ID = ka.get('id') or (ka.get('name') or '').split('/')[-1]
            print(f"Reusing existing KA: {KA_ID}")
            break
except Exception as e:
    print(f"List KA failed (non-fatal): {e}")

if not KA_ID:
    body = {
        "display_name": KA_NAME,
        "description": "DISA CTI over CISA advisories, DoD STIGs, ATT&CK",
        "instructions": KA_SYSTEM_PROMPT,
    }
    res = w.api_client.do('POST', KA_BASE, body=body)
    KA_ID = res.get('id') or (res.get('name') or '').split('/')[-1]
    print(f"Created KA: {KA_ID}")

# Endpoint name follows the convention ka-<short-id>-endpoint.
ka = w.api_client.do('GET', f'{KA_BASE}/{KA_ID}')
KA_ENDPOINT = ka.get('endpoint_name') or f"ka-{KA_ID[:8]}-endpoint"
print(f"KA endpoint: {KA_ENDPOINT}")

# Attach the corpus volume as a knowledge source if not already attached.
src_list = w.api_client.do('GET', f'{KA_BASE}/{KA_ID}/knowledge-sources').get('knowledge_sources', []) or []
have_corpus = any((s.get('files') or {}).get('path') == KA_VOLUME for s in src_list)
if not have_corpus:
    src_body = {
        "display_name": "corpus",
        "description": "CISA advisories + DoD STIG snippets + ATT&CK techniques.",
        "source_type": "FILES",
        "files": {"path": KA_VOLUME},
    }
    w.api_client.do('POST', f'{KA_BASE}/{KA_ID}/knowledge-sources', body=src_body)
    print(f"Attached knowledge source: {KA_VOLUME}")
else:
    print(f"Corpus volume already attached.")

# Persist for module 5 to pick up automatically.
cfg_set('ka_id', KA_ID)
cfg_set('ka_endpoint', KA_ENDPOINT)

print(f"\nKA_ID       = {KA_ID}")
print(f"KA_ENDPOINT = {KA_ENDPOINT}")
print("\nKA endpoint warm-up takes 10-20 minutes; module 5 tolerates a not-yet-ready KA.")
